In [ ]:
from transformers import CLIPProcessor, CLIPModel, AutoModel, AutoProcessor, SiglipProcessor, AutoImageProcessor
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from typing import List, Tuple, Optional, Union, Literal, Dict
from PIL import Image
import torch
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

import matplotlib.pyplot as plt
from torchvision import transforms
import seaborn as sns
import json
import pandas as pd
import math
import zipfile
!pip install h5py
import h5py

from time import sleep
import requests
from pathlib import Path
!pip install fake_useragent
from fake_useragent import UserAgent
!pip install natsort
from natsort import natsorted

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

## Использование предобученных мультимодальных моделей в качестве предобученных для получения эмбеддингов картин

## Предобработка данных для Christies

In [ ]:
from string import ascii_letters
christies_Gdrive_folder = '/content/drive/MyDrive/Сhristies_splited_full'

total_bukvi = ascii_letters[:ascii_letters.find('w') + 1]
parts_to_unzip = [f'Art_analytics_christies_spliteda' + letter for letter in total_bukvi]
output_zip = '/content/reconstructed.zip'
with open(output_zip, 'wb') as outfile:
    for fname in tqdm(parts_to_unzip, desc='анзипаю файлики'):
        filepath = os.path.join(christies_Gdrive_folder, fname)
        with open(filepath, 'rb') as infile:
            outfile.write(infile.read())

print(f"Единый зип файл: {output_zip} ({os.path.getsize(output_zip)/1024**3:.2f} GB)")

extract_to = '/content/christies_dataset'
os.makedirs(extract_to, exist_ok=True)

try:
    with zipfile.ZipFile(output_zip, 'r') as zf:
        zf.extractall(extract_to)
    print(f"Успешно распаковано в: {extract_to}")
except zipfile.BadZipFile as e:
    print(f'Ошибка: файл не является валидным ZIP')

print('Все сплиты распакованы')

In [ ]:
!rm -rf reconstructed.zip

In [ ]:
import gc
gc.collect()

In [ ]:
!wget -O metadata.csv https://drive.google.com/uc?id=1Ma1vA6KhhkVtuJ2NcqRVNhI3hDeRb2O4

In [ ]:
metadata = pd.read_csv(r'/content/metadata.csv')

In [ ]:
metadata.head()

## Предобработка данных для Sotheby's

In [ ]:
!wget -O data_sothebys.json https://drive.google.com/uc?id=1r-9IAgkvCS4s-BBpMYiPcNI5czO5PuNN

In [ ]:
with open(r'/content/data_sothebys.json', 'r', encoding='utf-8') as read_file:
  json_sothebys_data = json.load(read_file)

# json_sothebys_data

In [ ]:
df_sothebys = pd.json_normalize(json_sothebys_data,
                                record_path='lots',
                                meta=['auction_name', 'auction_city', 'auction_date'])
df_sothebys = df_sothebys[['auction_name', 'auction_city', 'auction_date', 'estimate_price',
             'lot_link', 'image_url']]
df_sothebys.sample(5)

In [ ]:
def date_encoder(x):
  split_dates = x.replace('–', ' ').lower().split(' ')
  if len(split_dates) == 5:
    return f'{split_dates[2]}/{split_dates[3]}/{split_dates[4]}'
  elif len(split_dates) == 3:
    return f'{split_dates[0]}/{split_dates[1]}/{split_dates[2]}'
  elif len(split_dates) == 4:
    return f'{split_dates[1]}/{split_dates[2]}/{split_dates[3]}'
  else:
    return x

In [ ]:
df_sothebys[['lower_estimate', 'upper_estimate']] = df_sothebys['estimate_price'].str.split(' - ', expand=True)
df_sothebys[['upper_estimate', 'currency']] = df_sothebys['upper_estimate'].str.split(' ', expand=True)
df_sothebys.drop(columns=['estimate_price'], inplace=True)
df_sothebys['upper_estimate'] = df_sothebys['upper_estimate'].str.replace(',', '').astype('float')
df_sothebys['lower_estimate'] = df_sothebys['lower_estimate'].str.replace(',', '').astype('float')
df_sothebys['auction_date'] = pd.to_datetime(df_sothebys['auction_date'].apply(lambda x: date_encoder(x)), format='%d/%B/%Y')


In [ ]:
df_sothebys

## Структура данных


Для корректной работы последующих операций по получению эмбеддингов картинок предполагается следующая изначальная структура данных:

![image.png](attachment:image.png)

Корневая папка может быть не обязательно папкой всего проекта, скорее, это папка всех данных

In [ ]:
def download_images(
    urls: Union[List, np.ndarray],
    base_dir: str = "data",
    batch_size: int = 100,
    timeout: float = 2,
    max_retries: int = 3,
) -> None:
    """
    DФункция для скачивания изображений из массива ссылок

    Принимает на вход массив ссылок на картинки и скачивает их в папку
    с названием base_dir

    Args:

        urls:  список или нампай массив ссылок на картинки
        base_dir: название базовой директории для хранения
        batch_size: размер одной папки с картинами
        timeout: задержка между попытками и время ожидания ответа запроса
        max_retries: максимальное количество попыток

    Returns: None

    """

    base_path = Path(base_dir)
    base_path.mkdir(exist_ok=True)

    for idx, url in enumerate(tqdm(urls, desc="Скачивание изображений картин")):
        try:
            batch_idx = idx // batch_size
            batch_dir = base_path / f"batch_{batch_idx}"
            batch_dir.mkdir(exist_ok=True)

            file_path = batch_dir / f"image_{idx}.jpg"

            if file_path.exists():
                print(f"Файл {file_path} уже существует. Пропускаем.")
                continue

            success = False

            for attempt in range(max_retries):
                try:
                    response = requests.get(
                        url=url,
                        headers={"User-Agent": UserAgent().random},
                        timeout=timeout,
                    )
                    if response.status_code == 200:
                        with open(file=file_path, mode="wb") as write_file:
                            write_file.write(response.content)
                            success = True
                        break
                    else:
                        print(f"У {idx} ошибка {response.status_code}: {url}")
                        continue
                except requests.RequestException as error:
                    print(f"Ошибка при скачивании по {url}\n{error}")
                    sleep(1)
                    pass
            if not success:
                print(f"Не удалось скачать {url}")
                continue

            sleep(0.5)
        except Exception as main_error:
            print(f"По адресу {url} неожиданная ошибка: {main_error}")

    print(f"Скачивание завершено. Всё загружено в {base_dir}")

    return None

In [ ]:
download_images(df_sothebys['image_url'].values, '/content/data', 20, 2, 3)

## Пайплайн получения эмбеддингов картин

In [ ]:
class AuctionsImageDataset(Dataset):
    '''Свой датасет для подгрузки картин c аукционов
    Возваращает по индексу кортеж: (картинка, название аукциона, полный путь к картинке)
    '''
    def __init__(self, root_dir: str,
                 extensions=(".png", ".jpg", ".jpeg", ".webp"),
                 max_fold_read: int =150) -> Optional[Tuple[torch.Tensor, str, str]]:
        super().__init__()
        self.root_dir = root_dir
        self.extensions = extensions
        self.image_paths = []
        self.auction_names = []
        self.max_fold_read = max_fold_read

        self.current_read = 0
        self.total_images = 0
        for auction_name in sorted(os.listdir(self.root_dir)):
          auction_path = os.path.join(self.root_dir, auction_name)
          if not os.path.isdir(auction_path):
              continue
          self.current_read += 1
          for file_name in sorted(os.listdir(auction_path)):
              if file_name.lower().endswith(self.extensions):

                  full_path = os.path.join(auction_path, file_name)
                  try:
                      with Image.open(full_path) as image:
                          image.verify()
                          self.total_images += 1
                      self.image_paths.append(full_path)
                      self.auction_names.append(auction_name)
                  except Exception as erorr:
                      print(f"Error loading image {full_path}: {erorr}")
                      continue
          if self.current_read >= self.max_fold_read:
            break
        print(f'Всего {self.total_images} картин')

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_full_path = self.image_paths[idx]
        auction_name = self.auction_names[idx]
        try:
            image = Image.open(image_full_path).convert("RGB")

            return image, auction_name, image_full_path
        except Exception as error:
            print(error)
            raise RuntimeError('рантайм который в датасете')

def collate_fn(batch):
    valid_batch = [b for b in batch if b is not None]
    if not valid_batch:
        return [], [], []

    images = []
    auction_names = []
    paths = []

    for img, name, path in valid_batch:
        images.append(img)
        auction_names.append(name)
        paths.append(path)

    return images, auction_names, paths

In [ ]:
def plot_images(dataloader, n=8, figsize=(12, 8)):
    batch = next(iter(dataloader))

    images, auction_names, image_paths = batch

    images = images[:n]
    auction_names = auction_names[:n]
    image_paths = image_paths[:n]

    cols = min(4, n)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1 and cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i in range(n):
        ploxoi_image = images[i]

        if isinstance(ploxoi_image, Image.Image):
            img_np = np.array(ploxoi_image)
        elif hasattr(ploxoi_image, 'permute'):
            img_np = ploxoi_image.permute(1, 2, 0).cpu().numpy()
        else:
            raise ValueError('Проблема с форматом картинки')
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

        axes[i].imshow(img_np)
    plt.tight_layout()
    plt.show()

In [ ]:
data_root = Path(r"/content/christies_dataset/images")
image_dataset = AuctionsImageDataset(root_dir=data_root)
image_dataloader = DataLoader(
    image_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn)

In [ ]:
plot_images(image_dataloader)

In [ ]:
class GoogleEmbeddings:
    '''
    Класс для получения эмбеддингов изображений с помощью предобученной модели от
    Google (Siglip).
    В качестве датасета используется кастомная реализация класса AuctionsImageDataset

    '''
    def __init__(
    self,
    model_name: str="google/siglip-base-patch16-224",
    device: Optional[str] = None,
    num_workers: int = 0) -> None:
      self.device = ('cuda' if torch.cuda.is_available() else 'cpu')
      self.model = AutoModel.from_pretrained(model_name).to(self.device)
      self.num_workers = num_workers
      self.processor = AutoProcessor.from_pretrained(model_name)
      self.model.eval()

    @torch.no_grad()
    def _get_embedding_(
        self,
        data_root: str,
        batch_size: int=16,
        shuffle: bool=False,
        norm_l2: bool=True,
        output_dir: str = './temp_embeddings'
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:

      '''Функция для формирования эмбеддингоd всех поданных изображений картин'''

      images_dataset = AuctionsImageDataset(data_root)

      images_dataloader = DataLoader(
          dataset=images_dataset,
          shuffle=False,
          batch_size=batch_size,
          pin_memory=True,
          collate_fn=collate_fn,
          drop_last=False,
          num_workers=self.num_workers
      )

      os.makedirs(output_dir, exist_ok=True)
      temp_emb_file = os.path.join(output_dir, 'embeddings.h5')
      temp_meta_file = os.path.join(output_dir, 'metadata.npz')

      with h5py.File(temp_emb_file, 'w') as write_file:
          max_shape = (None, 768)
          dataset_embeds = write_file.create_dataset(
              'embeddings',
              shape=(0, 768),
              maxshape=max_shape,
              chunks=True,
              dtype='float32'
          )
          all_auction_names = []
          all_image_paths = []
          offset = 0

          for batch_images, batch_auction_names, batch_image_paths in tqdm(images_dataloader, desc="Обработка картин", leave=True):
              if batch_images is None or len(batch_images) == 0:
                  continue

              images_preprocessed = self.processor(
                  images=batch_images,
                  return_tensors='pt',
                  padding=True,
                  do_rescale=True
              ).to(self.device)


              images_embeddings = self.model.get_image_features(**images_preprocessed)
              images_embeddings = images_embeddings.pooler_output
              if norm_l2:
                if isinstance(images_embeddings, torch.Tensor):
                  images_embeddings = torch.nn.functional.normalize(images_embeddings, p=2, dim=-1)
                else:
                  images_embeddings = images_embeddings / (np.linalg.norm(images_embeddings, ord=2, axis=-1, keepdims=True) + 1e-8)

              embeddings_np = images_embeddings.cpu().numpy().astype('float32')

              new_size = offset + len(embeddings_np)
              dataset_embeds.resize(new_size, axis=0)
              dataset_embeds[offset:new_size] = embeddings_np

              all_auction_names.extend(batch_auction_names)
              all_image_paths.extend(batch_image_paths)

              offset = new_size

      np.savez_compressed(
          temp_meta_file,
          auction_names=all_auction_names,
          image_paths=all_image_paths
      )

      with h5py.File(temp_emb_file, 'r') as read_file:
          embeddings = read_file['embeddings'][:]
          metadata = np.load(temp_meta_file)
          auction_names_array = np.array(metadata['auction_names'], dtype='object')
          image_paths_array = np.array(metadata['image_paths'], dtype='object')

      return embeddings, auction_names_array, image_paths_array

## Снижение размерности полученных эмбеддингов

In [ ]:
def pca_process(
        embeddings: Union[np.ndarray, torch.Tensor],
        var_threshold: float=0.95,
        plot: bool=True
):
    '''Функция ищет оптимальное количество главных компонент, в
    зависимости от заданного порогового значения уровня дисперсии'''

    pca = PCA(n_components=None, random_state=42)
    pca.fit(embeddings)
    cumsum_var = np.cumsum(pca.explained_variance_ratio_)

    best_n_components = np.argmax(cumsum_var >= var_threshold) + 1

    print(f'Для объяснения {var_threshold*100}% дисперсии требуется {best_n_components} главных компонент')
    pca_best = PCA(n_components=best_n_components, random_state=42)
    reduced_embeddings = pca_best.fit_transform(embeddings)
    if plot:
        plt.plot(cumsum_var)
        plt.axhline(y=var_threshold, color='r', linestyle='--')
        plt.xlabel('Количество главных компонент')
        plt.ylabel('Кумулятивная доля объясненной дисперсии')
        plt.show()
    return reduced_embeddings

## Кластеризация

In [ ]:
def find_best_n_cluster(
        embeddings: Union[np.ndarray, torch.Tensor],
        k_range: int=range(20, 60),
        batch_size: int=None,
        metric: Literal['inertia', 'silhouette'] = 'silhouette',
        random_state: int=42
):

    k_clusters = list(k_range)

    if batch_size is None:
        batch_size = int(np.sqrt(len(embeddings))) + 1

    scores = []

    for k in tqdm(k_clusters, desc='Поиск оптимального числа кластеров'):
        k_means = MiniBatchKMeans(n_clusters=k,
                                  batch_size=batch_size,
                                 max_iter=350,
                                 n_init=7,
                                 random_state=random_state)
        labels = k_means.fit_predict(embeddings)

        if metric == 'inertia':
            score = k_means.inertia_
        elif metric == 'silhouette':
            score = silhouette_score(embeddings,
                                    labels,
                                    sample_size=int(len(embeddings)*0.80),
                                    random_state=random_state)
        else:
            raise ValueError('Неизвестная метрика')

        scores.append(score)
    if metric == 'inertia':
        first_diff = np.diff(scores)
        second_diff = np.diff(first_diff)
        best_k_idx = np.argmax(second_diff) + 1
        best_k = k_clusters[best_k_idx]
    else:
        best_k_idx = np.argmax(scores)
        best_k = k_clusters[best_k_idx]

    return best_k

In [ ]:
def clusterisation(
        embeddings: Union[np.ndarray, torch.Tensor],
        n_clusters: int=None,
        k_range: int=range(20, 60),
        use_miniBatch: bool=True,
        batch_size: Optional[int] = 1024,
        random_state: int=42,
        verbose: bool=True
) -> dict:

    '''
    Пайплайн кластеризации

    Args:
        embeddings: np.ndarray or torch.Tensor
        n_clusters: int
        k_range: range
        use_miniBatch: bool
        random_state: int
        verbose: отображать ли итоги кластеризации текстом

    Returns:
        dict: словарь с кластерами и их центрами

    '''

    if batch_size is None:
      batch_size = int(np.sqrt(len(embeddings))) + 1

    if n_clusters is None:
        n_clusters = find_best_n_cluster(embeddings,
                                         k_range,
                                         random_state=random_state,
                                         batch_size=batch_size)
    if use_miniBatch:
        cluster_model = MiniBatchKMeans(n_clusters=n_clusters,
                                        random_state=random_state,
                                        batch_size=batch_size,
                                        max_iter=350,
                                        n_init=10)

    else:
        cluster_model = KMeans(n_clusters=n_clusters,
                                random_state=random_state,
                                n_init=5,
                                max_iter=100)
    labels = cluster_model.fit_predict(embeddings)
    inertia = cluster_model.inertia_

    centers = cluster_model.cluster_centers_
    distances = np.array([
        np.linalg.norm(embeddings[i] - centers[labels[i]])
        for i in range(len(embeddings))
    ])

    if verbose:
        print(f'Количество кластеров: {n_clusters}')
        print(f'Среднее внутрикластерное расстояние: {inertia}')
        print(f'Кластеризация завершена')

    return labels, distances

## Извлечение дополнительных визуальных фичей картин

In [ ]:
def get_visual_features_single(image_path: str) -> Optional[Dict]:
    """
    Извлекает визуальные признаки из одного изображения.
    """
    try:
        with Image.open(image_path) as image:
            image = Image.open(image_path).convert('RGB')
            w, h = image.size
            img_np = np.array(image, dtype=np.float32)

            R = img_np[:, :, 0]
            G = img_np[:, :, 1]
            B = img_np[:, :, 2]

            # Метрика Хасслера-Сусструнка
            rg = np.absolute(R - G)
            yb = np.absolute(0.5 * (R + G) - B)

            std_root = np.sqrt(np.std(rg)**2 + np.std(yb)**2)
            mean_root = np.sqrt(np.mean(rg)**2 + np.mean(yb)**2)

            colorfulness = std_root + (0.3 * mean_root)

            hsv_np = np.array(image.convert('HSV'), dtype=np.float32)
            saturation = np.mean(hsv_np[:, :, 1]) / 255.0
            brightness = np.mean(hsv_np[:, :, 2]) / 255.0

            gray_image = image.convert("L")
            contrast = np.std(np.array(gray_image)) / 255.0

            hist = np.array(gray_image.histogram())
            prob = hist / hist.sum()
            entropy = -np.sum(prob[prob > 0] * np.log2(prob[prob > 0]))

            warmth = (np.mean(R) - np.mean(B)) / 255.0

            del img_np, R, G, B, rg, yb, std_root, mean_root, hsv_np, gray_image, hist, prob

            return {
                'aspect_ratio': round(w / h, 2),
                'colorfulness': round(colorfulness, 2),
                'saturation': round(float(saturation), 3),
                'bright': round(float(brightness), 3),
                'contrast': round(float(contrast), 3),
                'entropy': round(float(entropy), 3),
                'warmth': round(float(warmth), 3),
                'image_path': image_path
            }

    except Exception as error:
        print(f'Ошибка при обработке {image_path}: {error}')
        return None

## Сбор итогового датафрейма для обучения модели бустинга

In [ ]:
def build_dataframe(all_image_paths: List[str],
                    embeddings: Union[np.ndarray, torch.Tensor],
                    max_workers: int = 2,
                    file_path: str =  None,
                    batch_size: int = 100,
                    get_features: bool = True) -> pd.DataFrame:

    if file_path is not None and embeddings is None:
        embeddings = np.load(file_path)
        print(4)

    if get_features:
      visual_features_list = []
      valid_image_paths = []

      total_paths = len(all_image_paths)

      for start_idx in tqdm(range(0, total_paths, batch_size),
                            desc='Обработка картинок батчами',
                            total=(total_paths + batch_size - 1) // batch_size):
          end_idx = min(start_idx + batch_size, total_paths)
          batch_paths = all_image_paths[start_idx:end_idx]

          batch_results = []

          with ThreadPoolExecutor(max_workers=max_workers) as executor:
              futures = [executor.submit(get_visual_features_single, path) for path in batch_paths]

              for future in as_completed(futures):
                  result = future.result()
                  if result is not None:
                      batch_results.append(result)
                      valid_image_paths.append(result['image_path'])
          visual_features_list.extend(batch_results)

          if len(visual_features_list) % (batch_size * 2) == 0:
              gc.collect()
              # pass

      if len(visual_features_list) == 0:
          raise ValueError("Не удалось извлечь признаки")

      if len(visual_features_list) != len(valid_image_paths):
          print('Ошибка размерности массива валидных ссылок и соответствующих признаков изображений')

      df_visual = pd.DataFrame(visual_features_list)
      del visual_features_list
      labels, distances = clusterisation(embeddings,
                                       use_miniBatch=True,
                                       batch_size=1024)
      df_final = df_visual.copy()
      df_final['labels'] = labels
      df_final['distances_to_center'] = distances
    else:
      labels, distances = clusterisation(embeddings,
                                       use_miniBatch=True,
                                       batch_size=1024)
      del embeddings
      df_final = pd.DataFrame(labels, columns=['labels'])
      df_final['distances_to_center'] = distances
      # df_final['labels'] = labels

    gc.collect()

    return df_final

In [ ]:
siglip_encoder = GoogleEmbeddings()

embeddings, all_auction_names, all_image_paths = siglip_encoder._get_embedding_(
    data_root=data_root,
    batch_size=16,
    shuffle=False,
    norm_l2=True
)

In [ ]:
data = np.load('/content/clear_embeddings_16934_siglip-base_no_l2norm.npy')
data.shape

In [ ]:
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

In [ ]:
pca_embeddings = pca_process(
    embeddings_scaled,
    var_threshold=0.9,
    plot=True
)

In [ ]:
np.save('/content/pca_reduced_embeddings_16934_siglip-base_l2norm_scaled.npy', pca_embeddings)
# np.save('/content/all_image_paths.npy', np.array(all_image_paths))

In [ ]:
df = build_dataframe(embeddings=None,
    all_image_paths=list(metadata['image_url'].values),
    file_path='/content/clear_embeddings_16934_siglip-base_no_l2norm.npy',
    get_features=False
)

In [ ]:
df['labels'].value_counts()

In [ ]:
df['image_url'] = np.load(r'/content/all_image_paths.npy', allow_pickle=True)
df['price'] = metadata.loc[:, 'image_price'].values
df.head()

In [ ]:
df = df.rename(columns={'image_path': 'image_url'})

## Визуализация кластеров (функция)

In [ ]:
def display_cluster_images(df: pd.DataFrame,
                           cluster_labels: str ,
                           image_paths: str,
                           distance_cluster: str,
                           cluster_number: int,
                           selection_method: str  = 'closest',
                           n_images: int= 10,
                           figsize: tuple = (15, 10)
                           ):
    '''
    Функция для отображения изображений из указанного кластера.

    Параметры:
    -----------
    cluster_labels : array-like
        Метки кластеров для каждого изображения
    image_paths : list
        Список путей к изображениям (должен соответствовать по длине cluster_labels)
    cluster_number : int
        Номер кластера для отображения
    n_images : int, optional (default=10)
        Количество изображений для отображения
    '''

    if selection_method not in ['closest', 'random', 'farthest']:
        print(f'Не реализованный метод сортировки изображений')
        return None

    cluster_df = df[df[cluster_labels] == cluster_number].copy()

    if len(cluster_df) == 0:
        print(f'В этом кластере нет изображений')
        return None

    if selection_method == 'closest':
        selected_df = cluster_df.sort_values(by=distance_cluster).head(n_images)
        title_suffix = "Ближайшие к центру кластера"
    elif selection_method == 'farthest':
        selected_df = cluster_df.sort_values(by=distance_cluster, ascending=False).head(n_images)
        title_suffix = "Наиболее удалённые от центра кластера"
    else:
        n_to_sample = min(n_images, cluster_df.shape[0])
        selected_df = cluster_df.sample(n=n_to_sample, random_state=42)
        title_suffix = f'Рандомные {n_images} картин кластера'

    n_cols = min(5, len(selected_df))
    n_rows = int(np.ceil(len(selected_df) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1 or n_cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()

    for ax in axes:
        ax.axis('off')

    for idx, (image_idx, row) in enumerate(selected_df.iterrows()):
        img_path = row[image_paths]

        try:
            if os.path.exists(img_path):
                img = Image.open(img_path)
                axes[idx].imshow(img)
                # axes[idx].set_title(f'Расстояние: {row[distance_cluster]:.3f}\n{str(os.path.basename(img_path)).replace('.jpg', '')}', fontsize=10)
                axes[idx].set_title(f'Расстояние: {row[distance_cluster]:.3f}', fontsize=10)
                axes[idx].axis('on')
            else:
                axes[idx].text(0.5, 0.5, 'Картина не нашлась',
                             ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].axis('on')
        except Exception as e:
            axes[idx].text(0.5, 0.5, 'не загрузилось',
                         ha='center', va='center', transform=axes[idx].transAxes)
            axes[idx].axis('on')

    fig.suptitle(f'Кластер {cluster_number} - {title_suffix}',
                 fontsize=14, y=0.98)

    plt.tight_layout()
    plt.show()
    print(f"Кластер {cluster_number}: {len(cluster_df)} изображений")
    print(f"Отображено: {n_images} изображений")

### Визуализация

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       11,
                       'random',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       16,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       21,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       1,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       17,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       12,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       2,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       19,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       3,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       18,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       6,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       10,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       15,
                       'random',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       0,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       7,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       14,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       13,
                       'closest',
                       20
                       )

In [ ]:
display_cluster_images(df,
                       'labels',
                       'image_url',
                       'distances_to_center',
                       9,
                       'closest',
                       20
                       )

## Сохранение итогов кластеризации

In [ ]:
df.to_csv(r'/content/siglip-base_16934_clustered_df.csv')

## Ансамблевая модель прогнозирования стоимости картины

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
def load_data(filepath: str) -> pd.DataFrame:
    '''
    Функция для загрузки датафрейма со всеми входными признаками и целевой переменной .csv формата
    '''
    df = pd.read_csv(filepath, index_col=0)
    # display(df.head(2))

    # required_columns = ['saturation',
    #                     'bright',
    #                     'contrast',
    #                     'entropy',
    #                     'temeparature',
    #                     'labels',
    #                     'distance_to_center',
    #                     'price']

    # for column in required_columns:
    #     if column not in df.columns:
    #         print(f'Отсутствует столбец {column} либо он имеет иное название')
    #         raise ValueError('Не соответствие колонок в датафрейме')

    if df.isna().any().any():
        raise ValueError('В датафрейме есть пропуски')

    return df

In [ ]:
def prepare_features(df: pd.DataFrame):
    if 'price' not in df.columns:
        raise ValueError('Отсутствует колонка цены под названием <price>')

    target = np.log(df['price'].copy().values)
    features = df.drop(columns=['price', 'image_url']).copy()

    object_features = []
    for column in df.select_dtypes(include=['object']).columns:
        object_features.append(column)

    if 'cluster' in object_features:
        features['cluster'] = features['cluster'].astype('category')

    return features, target, object_features

In [ ]:
def data_split(features: pd.DataFrame,
                     target: pd.Series | pd.DataFrame,
                       random_state: int=42):
    # print(f"Features shape: {features.shape}")
    # print(f"Target shape: {target.shape}")
    return train_test_split(features, target, random_state=random_state, test_size=0.3)

In [ ]:
def hyper_tunnning(X_train: pd.DataFrame | np.ndarray,
                  y_train: pd.DataFrame | np.ndarray,
                  object_features: list,
                  model_type: str = 'sklearn',
                  cv: int = 5,
                  random_state: int = 42):
    if model_type == 'catboost':
        # model = CatBoostClassifier(
            # iterations=1000)
        #     learning_rate=0.1,
        #     depth=6,
            # loss_function='Logloss',
        #     eval_metric='AUC',
        # )
        raise NotImplementedError('четааа дааа проблема')

    else:
        model = GradientBoostingRegressor()

    param_grid = {
        'max_depth': [2, 4, 6, 8,],
        'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.25],
        'n_estimators': [50, 100, 150, 200, 300, 350],
        'subsample': [0.75, 0.8, 0.9, 1.0],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2'],
        'criterion': ['squared_error', 'friedman_mse']
            }
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=50,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=0,
        refit=True,
        random_state=random_state
    )

    random_search.fit(X_train, y_train)

    return random_search.best_estimator_, random_search.best_params_


In [ ]:
def train_model(X_train: pd.DataFrame,
                y_train: pd.DataFrame,
                best_params: dict,
                object_features: list):
    '''
    Функция для итогового обучения модели
    '''

    final_model = GradientBoostingRegressor(
        **best_params,
        random_state=42)
    final_model.fit(X_train, y_train)
    return final_model

In [ ]:
def evalute_model(model: GradientBoostingRegressor,
                  X_test: pd.DataFrame,
                  y_test: pd.Series) -> Dict[str, float]:
    y_pred = model.predict(X_test)

    metrics = {
        'mae': mean_absolute_error(y_test, y_pred),
        'mse': mean_squared_error(y_test, y_pred),
        'rmse': root_mean_squared_error(y_test, y_pred)
    }
    return metrics

In [ ]:
import joblib
def save_model(model: GradientBoostingRegressor, filepath: str):
    joblib.dump(model, filepath)

In [ ]:
def main_pipeline(data_path: str, model_save_path: str=r'/content/save_models/picture_price_model1.pkl'):
    '''
    Функция для полного запуска пайплайна бустинга
    '''

    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    df = load_data(data_path)
    features, target, object_features = prepare_features(df)

    X_train, x_test, y_train, y_test = data_split(features, target)
    best_model, best_params = hyper_tunnning(X_train, y_train, object_features)
    print(f'Лучшие параметры: {best_params}')

    final_model = train_model(X_train, y_train,
                              best_params, object_features)
    metrics = evalute_model(final_model, x_test, y_test)
    print(f'Итоговые метрики на тестовой выборке: {metrics}')

    save_model(final_model, model_save_path)

    return final_model, metrics

In [ ]:
sns.histplot(data=df, x='price')

In [ ]:
sns.histplot(np.log(df['price'].values))

In [ ]:
modeeel, metr = main_pipeline(data_path=r'/content/siglip-base_15200_clustered_df.csv')